# Шаг 13. Сквозной конвейер обработки данных

В этом блокноте используется общий модуль `src/pipeline.py`.
Все этапы обработки выполняются функциями модуля, чтобы результаты
блокнота и запуска из командной строки были согласованы.
Для работы используется ранее очищенный набор данных в формате Parquet.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

# Корень проекта — на уровень выше папки notebooks
PROJECT_DIR = Path.cwd().parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src import pipeline

SOURCE_PATH = Path(r"E:\laba_results\clean_final.parquet")
OUTPUT_PATH = Path(r"E:\laba_results\step13_notebook.parquet")

print("Корень проекта:", PROJECT_DIR)
print("Исходный файл:", SOURCE_PATH)
print("Исходный файл существует:", SOURCE_PATH.is_file())

Корень проекта: c:\Users\мирон\Desktop\laba po pitonu
Исходный файл: E:\laba_results\clean_final.parquet
Исходный файл существует: True


## Этап 1. Очистка данных

Вызывается общая функция clean_data из модуля src/pipeline.py.
Она преобразует дату и время, удаляет полные дубликаты и сохраняет результат
в отдельный Parquet-файл. Исходный файл не изменяется.

In [2]:
result_cleaning = pipeline.clean_data(
    source_path=SOURCE_PATH,
    output_path=OUTPUT_PATH,
    mode="full",
)

display(pd.DataFrame([{
    "этап": "Очистка данных",
    "строк до": result_cleaning["rows_before"],
    "строк после": result_cleaning["rows_after"],
    "изменено строк": result_cleaning["changed"],
    "изменения": "Преобразование даты и удаление полных дубликатов",
    "файл результата": result_cleaning["output"],
}]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,этап,строк до,строк после,изменено строк,изменения,файл результата
0,Очистка данных,55418645,55418645,0,Преобразование даты и удаление полных дубликатов,E:\laba_results\step13_notebook.parquet


## Этап 2. Анализ пропущенных значений

Общая функция analyze_missing из модуля src/pipeline.py
подсчитывает пропуски в каждом столбце и их процент от общего числа строк.
Результат сохраняется в отдельный CSV-файл. Исходный набор не изменяется.

In [3]:
MISSING_PATH = Path(r"E:\laba_results\step13_missing.csv")

missing_df = pipeline.analyze_missing(
    source_path=OUTPUT_PATH,
    report_path=MISSING_PATH,
)

display(missing_df)

print("Строк до анализа:", len(pd.read_parquet(OUTPUT_PATH)))
print("Строк после анализа:", len(pd.read_parquet(OUTPUT_PATH)))
print("Изменено строк: 0")
print("Отчёт сохранён:", MISSING_PATH)

,столбец,пропусков,доля_пропусков_процентов
0,key,0,0.0000
1,fare_amount,0,0.0000
2,pickup_datetime,0,0.0000
3,pickup_longitude,0,0.0000
4,pickup_latitude,0,0.0000
5,dropoff_longitude,376,0.0007
6,dropoff_latitude,376,0.0007
7,passenger_count,0,0.0000


Строк до анализа: 55418645
Строк после анализа: 55418645
Изменено строк: 0
Отчёт сохранён: E:\laba_results\step13_missing.csv


## Этап 3. Анализ выбросов

Выбросы в столбце fare_amount анализируются двумя методами:
межквартильным размахом (IQR) и Z-score. На этом этапе выбросы
только выявляются и подсчитываются, но не удаляются.

In [4]:
OUTLIERS_PATH = Path(r"E:\laba_results\step13_outliers.csv")

outliers_df = pipeline.analyze_outliers(
    source_path=OUTPUT_PATH,
    report_path=OUTLIERS_PATH,
)

display(outliers_df)

print("Строк до анализа:", 55_418_645)
print("Строк после анализа:", 55_418_645)
print("Изменено строк: 0")
print("Отчёт сохранён:", OUTLIERS_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,столбец,метод,обнаружено_выбросов
0,fare_amount,IQR,4773101
1,fare_amount,Z-score,1506601


Строк до анализа: 55418645
Строк после анализа: 55418645
Изменено строк: 0
Отчёт сохранён: E:\laba_results\step13_outliers.csv


In [5]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

BASE_DIR = Path(r"E:\laba_results")

files = [
    BASE_DIR / "clean_final.parquet",
    BASE_DIR / "step13_notebook.parquet",
    BASE_DIR / "step13_missing.csv",
    BASE_DIR / "step13_outliers.csv",
    BASE_DIR / "step13_missing_analysis.csv",
    BASE_DIR / "step13_outliers.csv",
]

for path in files:
    print(f"\nФайл: {path.name}")
    print("Существует:", path.exists())

    if path.exists() and path.suffix == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        print("Строк:", f"{metadata.num_rows:,}")
        print("Размер, МБ:", round(path.stat().st_size / 1024**2, 2))


Файл: clean_final.parquet
Существует: True
Строк: 55,418,645
Размер, МБ: 2718.61

Файл: step13_notebook.parquet
Существует: True
Строк: 55,418,645
Размер, МБ: 2718.82

Файл: step13_missing.csv
Существует: True

Файл: step13_outliers.csv
Существует: True

Файл: step13_missing_analysis.csv
Существует: False

Файл: step13_outliers.csv
Существует: True
